In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pandas as pd
from tqdm import tqdm
import traceback

torch.cuda.empty_cache()

In [2]:
device = "cuda"  # the device to load the model onto

model_name = "speakleash/Bielik-11B-v2.2-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [3]:
model.to(device)

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32128, 4096)
    (layers): ModuleList(
      (0-49): 50 x MistralDecoderLayer(
        (self_attn): MistralSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): MistralRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm()
        (post_attention_layernorm): MistralRMSNorm()
      )
    )
    (norm): MistralRMSNorm(

In [4]:
footprint_in_bytes = model.get_memory_footprint()
footprint_in_gigabytes = footprint_in_bytes / 1024**3
print(f"Model size: {footprint_in_gigabytes:.2f} GB")

Model size: 20.80 GB


In [5]:
relations_schema = pd.read_csv("data/edc_baseline/schema.csv", header=None)

relations_description = ""
for i in range(0, len(relations_schema)):
    relation_name, relation_description = relations_schema.iloc[i]
    relations_description += f"{relation_name} - {relation_description}\n"

zawód - Poprzednik wykonuje pracę w zawodzie określonym przez następnik.
członek - Poprzednik jest członkiem organizacji określonej przez następnik.
miejsce urodzenia - Poprzednik urodził się w miejscu określonym przez następnik.
lokalizacja - Poprzednik znajduje się w miejscu określonym przez następnik.
typ - Poprzednik jest rodzaju określonego przez następnik.
siedziba - Siedziba poprzednika znajduje się w miejscu określonym przez następnik.
liczba ludności - Poprzednik ma liczbę ludności określoną przez następnik.
rodzic - Poprzednik jest rodzicem następnika.
z kraju - Poprzednik pochodzi z kraju określonego przez następnik.
film ma reżysera - Film (poprzednik) ma reżysera określonego przez następnik.
lider lokalizacji - Poprzednik jest liderem lokalizacji określonej przez następnik.
rok wydarzenia - Poprzednik wydarzył się w roku określonym przez następnik.
gatunek - Poprzednik należy do gatunku określonego przez następnik.
organizacja ma członka - Organizacja (poprzednik) ma człon

In [6]:
with open("data/edc_baseline/oie_few_shot_examples.txt", "r") as file:
    fewshot_examples = "".join(file.readlines())

"Przykład 1:\nTekst: Diego Serrano (ur. 5 lutego 1973 w Quito, w Ekwadorze) – amerykański aktor telewizyjny i filmowy .\nTrójka: [['Diego Serrano', 'zawód', 'Aktor']]\n\nPrzykład 2:\nTekst: Manifesto – szósty album studyjny brytyjskiej grupy rockowej Roxy Music, wydany w 1979 roku nakładem E.G.\nTrójka: [['Manifest', 'typ', 'Album studyjny']]\n\nPrzykład 3:\nTekst: Cleve – miasto w Australii, w stanie Australia Południowa.\nTrójka: [['Cleve', 'lokalizacja', 'Australia']]\n\nPrzykład 4:\nTekst: Fahad Al-Mehallel (ur. Al-Mehallel przez 9 lat bronił barw Al-Shabab Rijad.\nTrójka: [['Fahad Al-Mehallel', 'miejsce urodzenia', 'Rijad']]\n\nPrzykład 5:\nTekst: Oscar Martínez (ur. 23 października 1949 w Buenos Aires) – argentyński aktor filmowy, teatralny i telewizyjny, również dramaturg i reżyser teatralny.\nTrójka: [['Oscar Martínez', 'miejsce urodzenia', 'Buenos Aires']]\n\nPrzykład 6:\nTekst: Ireneusz Krosny (ur. 31 marca 1968 w Tychach) – polski aktor, mim, specjalizujący się w pantomimie 

In [7]:
with open("data/edc_baseline/dataset.txt", "r") as file:
    dataset = file.readlines()

['Richard Herd, właściwie Richard Thomas Herd Jr. (ur. 26 maja 2020 w Los Angeles) – amerykański aktor filmowy, telewizyjny i głosowy.\n',
 'Saint Clement (jèrriais Saint Cliément, fr. Saint-Clément) - miasto na wyspie Jersey (Wyspy Normandzkie); 8 403 mieszkańców (2008) & men=gcis & lng=en & des=gamelan & geo=-4 & srt=pnan & col=abcdefghimoq & msz=1500 & geo=-238 World Gazetteer Ośrodek przemysłowy.\n',
 'DJ Feel-X właściwie Sebastian Filiks (ur. 7 lutego 1978 w Lublinie ) – polski DJ i producent muzyczny .\n',
 'Sean Kanan, właściwie Sean Perelman (ur. 2 listopada 1966 w Cleveland) – amerykański aktor telewizyjny i filmowy .\n',
 'mały|Dom rodzinny Hanny Schygulli w Chorzowie Starym mały|Hanna Schygulla (1982) Hanna Schygulla (ur. 25 grudnia 1943 w Chorzowie Starym) – niemiecka aktorka filmowa, teatralna i telewizyjna.\n']

In [8]:
def create_prompt_content(relations_description, fewshot_examples, sample):
    return f"""Twoim zadaniem jest wyciągnąć jedną relację łączącą dwa obiekty występujące w tekście, jako trójkę. Trójka musi być w postaci [[Poprzednik, Relacja, Następnik]]. Poprzednik i Następnik są wyrażeniami zapisanymi w tekście. Relacja jest krótkim zapisem związku, jaki łączy Poprzednik i Następnik.
W swojej odpowiedzi przedstaw dokładnie jedną trójkę. Nie podawaj żadnych innych informacji czy wyjaśnień.
            
Relacje, które mogą wystąpić w tekście to:
{relations_description}

Poniżej przykłady zdań, w których występują obiekty, dla których należy wyciągnąć relację:
{fewshot_examples}

Teraz wyciągnij relację z poniższego tekstu:
Tekst: {sample}"""

In [9]:
def create_messages_template(relations_description, fewshot_examples, sample):

    return [
        {
            "role": "system",
            "content": "Odpowiadaj krótko, precyzyjnie i wyłącznie w języku polskim. Nie udzielaj żadnych wyjaśnień.",
        },
        {
            "role": "user",
            "content": create_prompt_content(
                relations_description, fewshot_examples, sample
            ),
        },
    ]

In [10]:
def generate_response(input_ids):
    with torch.no_grad():
        input_ids = input_ids.to(device)
        generated_ids = model.generate(input_ids, max_new_tokens=1000, do_sample=True)
        decoded = tokenizer.batch_decode(generated_ids)
        return decoded[0]

In [11]:
def extract_model_response(decoded):
    beginning = "<|im_start|> assistant"
    end = "<|im_end|>"
    return decoded.split(beginning)[-1].split(end)[0].strip()

In [12]:
responses = []
errors = []

DEBUG = False
for sample in tqdm(dataset[0:5] if DEBUG else dataset):
    try:
        print("Sample:", sample.strip())

        template = create_messages_template(
            relations_description, fewshot_examples, sample
        )

        input_ids = tokenizer.apply_chat_template(
            template, return_tensors="pt", add_generation_prompt=True
        )

        response = generate_response(input_ids)
        response = extract_model_response(response)
        responses.append(response + "\n")
        print("Response:", response)
    except Exception as e:
        tb = traceback.format_exc()
        error_message = f"Error for sample: {sample.strip()}\n{tb}\n"
        errors.append(error_message + "\n")
        responses.append("Error\n")
        print("Error:", error_message)

with open("data/edc_baseline/extracted_relations.txt", "w") as file:
    file.writelines(responses)

with open("data/edc_baseline/errors.txt", "w") as file:
    file.writelines(errors)

100%|██████████| 17171/17171 [00:00<00:00, 163935.70it/s]
